**Download necessary files**

In [1]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\wayne\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

**Import the necessary libraries**

In [4]:
import numpy as np
import random
import string

**Import libraries for scraping - beautifulSoup**

In [7]:
import bs4 as bs
import urllib.request
import re

**Read the web link**

In [10]:
raw_html = urllib.request.urlopen('https://en.wikipedia.org/wiki/Natural_language_processing')
raw_html = raw_html.read()

**Convert the web link contents into a file**

In [13]:
article_html = bs.BeautifulSoup(raw_html, 'lxml')

**Find only the html text containing inside < p > tags**

In [16]:
article_paragraphs = article_html.find_all('p')

**Read all the text inside paragraphs**

In [19]:
article_text = ''

for para in article_paragraphs:
    article_text += para.text
    
article_text

'Natural language processing (NLP) is a subfield of computer science and especially artificial intelligence. It is primarily concerned with providing computers with the ability to process data encoded in natural language and is thus closely related to information retrieval, knowledge representation and computational linguistics, a subfield of linguistics.\nMajor tasks in natural language processing are speech recognition, text classification, natural-language understanding, and natural-language generation.\nNatural language processing has its roots in the 1950s.[1] Already in 1950, Alan Turing published an article titled "Computing Machinery and Intelligence" which proposed what is now called the Turing test as a criterion of intelligence, though at the time that was not articulated as a problem separate from artificial intelligence. The proposed test includes a task that involves the automated interpretation and generation of natural language.\nThe premise of symbolic NLP is well-summ

**Split the sentences**

In [22]:
corpus = nltk.sent_tokenize(article_text)
corpus

['Natural language processing (NLP) is a subfield of computer science and especially artificial intelligence.',
 'It is primarily concerned with providing computers with the ability to process data encoded in natural language and is thus closely related to information retrieval, knowledge representation and computational linguistics, a subfield of linguistics.',
 'Major tasks in natural language processing are speech recognition, text classification, natural-language understanding, and natural-language generation.',
 'Natural language processing has its roots in the 1950s.',
 '[1] Already in 1950, Alan Turing published an article titled "Computing Machinery and Intelligence" which proposed what is now called the Turing test as a criterion of intelligence, though at the time that was not articulated as a problem separate from artificial intelligence.',
 'The proposed test includes a task that involves the automated interpretation and generation of natural language.',
 "The premise of sy

**Clean the text**

In [25]:
for i in range(len(corpus )):
    corpus [i] = corpus [i].lower()
    corpus [i] = re.sub(r'\W',' ',corpus [i])
    corpus [i] = re.sub(r'\s+',' ',corpus [i])

**Create a dictionary of word frequency**

In [28]:
wordfreq = {}
for sentence in corpus:
    tokens = nltk.word_tokenize(sentence)
    for token in tokens:
        if token not in wordfreq.keys():
            wordfreq[token] = 1
        else:
            wordfreq[token] += 1

**Filter the top 200 words**

In [31]:
import heapq
most_freq = heapq.nlargest(200, wordfreq, key=wordfreq.get)

**Compute IDF values**

In [34]:
word_idf_values = {}
for token in most_freq:
    doc_containing_word = 0
    for document in corpus:
        if token in nltk.word_tokenize(document):
            doc_containing_word += 1
    word_idf_values[token] = np.log(len(corpus)/(1 + doc_containing_word))

**Compute TF values**

In [36]:
word_tf_values = {}
for token in most_freq:
    sent_tf_vector = []
    for document in corpus:
        doc_freq = 0
        for word in nltk.word_tokenize(document):
            if token == word:
                  doc_freq += 1
        word_tf = doc_freq/len(nltk.word_tokenize(document))
        sent_tf_vector.append(word_tf)
    word_tf_values[token] = sent_tf_vector

**Compute TF-IDF values**

In [39]:
tfidf_values = []
for token in word_tf_values.keys():
    tfidf_sentences = []
    for tf_sentence in word_tf_values[token]:
        tf_idf_score = tf_sentence * word_idf_values[token]
        tfidf_sentences.append(tf_idf_score)
    tfidf_values.append(tfidf_sentences)

In [41]:
tf_idf_model = np.asarray(tfidf_values)

In [43]:
tf_idf_model = np.transpose(tf_idf_model)

In [45]:
tf_idf_model

array([[0.        , 0.00894022, 0.04542777, ..., 0.        , 0.        ,
        0.        ],
       [0.00678011, 0.00368127, 0.0374111 , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.03533271, ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.00794909, 0.00647396, 0.01096532, ..., 0.        , 0.        ,
        0.        ],
       [0.01002277, 0.01088375, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.00390718, 0.00212141, 0.04311788, ..., 0.        , 0.        ,
        0.        ]])

In [47]:
import pandas as pd
pd.DataFrame(tf_idf_model, columns=most_freq)

,the,of,and,language,in,to,a,natural,nlp,cognitive,...,gradual,lessening,dominance,chomskyan,theories,transformational,whose,theoretical,underpinnings,discouraged
0,0.000000,0.008940,0.045428,0.058451,0.000000,0.000000,0.068672,0.068672,0.074390,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.006780,0.003681,0.037411,0.024068,0.026097,0.044340,0.028277,0.028277,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.035333,0.136385,0.049295,0.000000,0.000000,0.160235,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.025614,0.000000,0.000000,0.090923,0.098589,0.000000,0.000000,0.106823,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.010722,0.002911,0.014790,0.000000,0.020635,0.000000,0.044717,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,0.028815,0.007823,0.039749,0.051144,0.000000,0.000000,0.060088,0.060088,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,0.014114,0.005109,0.012979,0.016700,0.000000,0.015383,0.039241,0.019621,0.042508,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,0.012807,0.006954,0.000000,0.045462,0.000000,0.000000,0.000000,0.053412,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,0.019210,0.005215,0.000000,0.068193,0.073942,0.000000,0.040059,0.040059,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9,0.023847,0.012948,0.021931,0.000000,0.030597,0.025992,0.000000,0.000000,0.000000,0.000000,...,0.097697,0.097697,0.097697,0.097697,0.097697,0.000000,0.000000,0.000000,0.000000,0.000000
